In [1]:
# Restart the runtime after this cell finishes, then run from the next cell onward
!pip install -q scanpy leidenalg igraph harmony-pytorch "pandas>=2.3" scikit-learn scikit-misc squidpy git+https://github.com/prabhakarlab/Banksy_py.git
!pip install -q scvi-tools cellcharter || echo "scvi-tools/cellcharter install failed; CellCharter will be skipped"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [2]:
DATA_PATH = "/content/global_z0.h5ad"  # adjustable
OUT_DIR = "/content/outputs/m2_consensus"  # adjustable
SEED = 0  # adjustable
SMOKE_TEST = False  # adjustable
SMOKE_N = 2000  # adjustable

BANKSY_LAMBDAS = [0.2, 0.5, 0.8]  # adjustable
BANKSY_SEEDS = [0, 1, 42]  # adjustable
K_GEOM = 30  # adjustable
BANKSY_RES = 0.3  # adjustable
BANKSY_PCS = 50
BANKSY_NUM_NN = 50

USE_CELLCHARTER = True  # adjustable
CC_N_LATENT = 10  # adjustable
CC_N_LAYERS = 3  # adjustable
CC_KNN = 15  # adjustable
CC_K_RANGE = (3, 12)  # adjustable
CC_MAX_RUNS = 10  # adjustable
SCVI_MAX_EPOCHS = 200  # adjustable

USE_M1_PARTITION = True  # adjustable
M1_PATH = "/content/outputs/m1_composition/labels.csv"  # adjustable

CONSENSUS_KNN = 50  # adjustable
RES_CANDIDATES = [round(0.1 * i, 1) for i in range(1, 11)]  # adjustable
N_BOOT = 10  # adjustable
BOOT_FRAC = 0.8  # adjustable
K_MIN = 4  # adjustable
K_MAX = 12  # adjustable
MIN_NICHE_FRAC = 0.005  # adjustable
LEIDEN_ITERS = 10  # adjustable
TOP_N_MARKERS = 10
BASELINE = {"n_niches": 7, "ARI": 0.248, "NMI": 0.323}

CFG = {k: v for k, v in dict(globals()).items() if k.isupper()}

In [3]:
import os, json, time, gc, random, itertools, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc, torch
import matplotlib.pyplot as plt
import igraph as ig, leidenalg as la
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score as ari_score, normalized_mutual_info_score as nmi_score, silhouette_score
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from harmony import harmonize
warnings.filterwarnings("ignore")
COORDS = ("cx_g", "cy_g", "spatial_global")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def load_smi(path, smoke=False, n_smoke=2000, seed=0):
    ad = sc.read_h5ad(path)
    ad = ad[:, ~ad.var_names.str.startswith("NegPrb")].copy()
    ad = ad[ad.obs["Region"].notna()].copy()
    ad.obs_names_make_unique()
    ad.obs["Region"] = ad.obs["Region"].astype(str)
    ad.obs["batch"] = ad.obs["batch"].astype(str)
    xy = np.asarray(ad.obsm["spatial_global"])
    ad.obs["cx_g"], ad.obs["cy_g"] = xy[:, 0], xy[:, 1]
    if smoke:
        keep = np.sort(np.random.RandomState(seed).choice(ad.n_obs, min(n_smoke, ad.n_obs), replace=False))
        ad = ad[keep].copy()
    ad.X = ad.X.astype(np.float32)
    ad.layers["counts"] = ad.X.copy()
    sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
    return ad

def run_banksy(ad, k_geom, lam, res, seed):
    bank = initialize_banksy(ad, COORDS, k_geom, nbr_weight_decay="scaled_gaussian", max_m=1,
        plt_edge_hist=False, plt_nbr_weights=False, plt_agf_angles=False, plt_theta=False)
    bank, _ = generate_banksy_matrix(ad, bank, [lam], max_m=1)
    seed_all(seed)
    pca_umap(bank, pca_dims=[BANKSY_PCS], add_umap=False, plt_remaining_var=False)
    sub = bank["scaled_gaussian"][lam]["adata"]
    seed_all(seed)
    sub.obsm["reduced_pc_50"] = harmonize(sub.obsm["reduced_pc_50"], sub.obs, batch_key="batch")
    out, _ = run_Leiden_partition(bank, [res], num_nn=BANKSY_NUM_NN, num_iterations=-1, partition_seed=1234, match_labels=False)
    plt.close("all")
    labels = [str(x) for x in out.loc[out.index[0], "labels"].dense]
    assert len(labels) == ad.n_obs
    pcs = np.asarray(sub.obsm["reduced_pc_50"])
    return pd.Series(labels, index=sub.obs_names).reindex(ad.obs_names), pcs

def top10_markers(ad, labels, top_n=10):
    mk = sc.AnnData(ad.layers["counts"].copy(), obs=pd.DataFrame({"niche": pd.Categorical(np.asarray(labels).astype(str))}, index=ad.obs_names), var=ad.var.copy())
    sc.pp.normalize_total(mk, target_sum=1e4); sc.pp.log1p(mk)
    sc.tl.rank_genes_groups(mk, "niche", method="wilcoxon")
    df = sc.get.rank_genes_groups_df(mk, group=None)
    df = df.rename(columns={"group": "niche", "names": "gene", "logfoldchanges": "log2FC", "scores": "score", "pvals_adj": "padj"})
    df["rank"] = df.groupby("niche").cumcount() + 1
    df = df[df["rank"] <= top_n][["niche", "rank", "gene", "log2FC", "score", "padj"]]
    df["niche"] = df["niche"].astype(str)
    return df.sort_values(["niche", "rank"], key=lambda s: s.astype(int) if s.name == "niche" else s).reset_index(drop=True)

def evaluate(labels, region):
    labels, region = np.asarray(labels).astype(str), np.asarray(region).astype(str)
    ct = pd.crosstab(labels, region); ct.index.name = "niche"
    ct = ct.loc[sorted(ct.index, key=int)]
    maj, pur = ct.idxmax(axis=1), ct.max(axis=1) / ct.sum(axis=1)
    return {"ARI": float(ari_score(region, labels)), "NMI": float(nmi_score(region, labels))}, ct, maj, pur

def spatial_knn(coords, batch, k):
    n, rows, cols, boxes = len(coords), [], [], []
    batch = np.asarray(batch); ub = np.unique(batch)
    for b in ub:
        ix = np.where(batch == b)[0]; kk = min(k, len(ix) - 1)
        if kk < 1: continue
        nn = NearestNeighbors(n_neighbors=kk + 1).fit(coords[ix])
        idx = nn.kneighbors(coords[ix], return_distance=False)[:, 1:]  # drop self
        rows.append(np.repeat(ix, kk)); cols.append(ix[idx].ravel())
        boxes.append((b, coords[ix].min(0), coords[ix].max(0)))
    n_over = sum(bool(np.all(a[1] < c[2]) and np.all(c[1] < a[2])) for a, c in itertools.combinations(boxes, 2))
    if n_over: print(f"WARNING: {n_over} batch pairs overlap in spatial_global coordinates")
    r, c = np.concatenate(rows), np.concatenate(cols)
    return sp.csr_matrix((np.ones(len(r), dtype=np.float32), (r, c)), shape=(n, n))

def state_clusters(pcs, res, n_neighbors=30, seed=0):
    t = sc.AnnData(np.asarray(pcs, dtype=np.float32))
    sc.pp.neighbors(t, n_neighbors=n_neighbors, use_rep="X", random_state=seed)
    sc.tl.leiden(t, resolution=res, random_state=seed, key_added="state")
    return t.obs["state"].astype(str).values

def gene_filters(ad, labels, thr=0.6):
    X = ad.layers["counts"]; labels = np.asarray(labels)
    frac = np.vstack([np.asarray((X[labels == g] > 0).mean(0)).ravel() for g in np.unique(labels)])
    return ad.var_names[(frac >= thr).all(0)].tolist()

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [4]:
def _js(o):
    return o.item() if hasattr(o, "item") else str(o)

def plot_vs_truth(ad, niche, path):
    fig, axs = plt.subplots(1, 2, figsize=(16, 7))
    for ax, vals, title in [(axs[0], ad.obs["Region"].values, "Expert Region"), (axs[1], np.asarray(niche), "Consensus niches")]:
        cats = sorted(set(vals), key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))
        cmap = plt.get_cmap("tab20")
        for i, c in enumerate(cats):
            m = vals == c
            ax.scatter(ad.obs["cx_g"].values[m], ad.obs["cy_g"].values[m], s=1, color=cmap(i % 20), label=c)
        ax.set_title(title); ax.set_aspect("equal"); ax.legend(markerscale=8, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)

def save_contract(out_dir, ad, niche, extra_cols, mk_long, summary, ct, metrics, config):
    os.makedirs(out_dir, exist_ok=True)
    lab = pd.DataFrame({"niche": np.asarray(niche).astype(str)}, index=ad.obs_names)
    for c in ["Region", "cx_g", "cy_g", "batch"]: lab[c] = ad.obs[c].values
    for k, v in extra_cols.items(): lab[k] = v
    lab.to_csv(f"{out_dir}/labels.csv")
    mk_long.to_csv(f"{out_dir}/niche_markers_long.csv", index=False)
    wide = mk_long.pivot(index="rank", columns="niche", values="gene")
    wide[sorted(wide.columns, key=int)].to_csv(f"{out_dir}/niche_markers_wide.csv")
    summary.to_csv(f"{out_dir}/niche_summary.csv", index=False)
    ct.to_csv(f"{out_dir}/crosstab_vs_region.csv")
    json.dump(metrics, open(f"{out_dir}/metrics.json", "w"), indent=2, default=_js)
    json.dump(config, open(f"{out_dir}/run_config.json", "w"), indent=2, default=_js)
    plot_vs_truth(ad, niche, f"{out_dir}/niche_vs_truth.png")

In [5]:
t_start = time.time()
seed_all(SEED)
out_dir = OUT_DIR + ("_smoke" if SMOKE_TEST else "")
os.makedirs(out_dir, exist_ok=True)
ad = load_smi(DATA_PATH, SMOKE_TEST, SMOKE_N, SEED)
n = ad.n_obs
coords = np.asarray(ad.obsm["spatial_global"], dtype=np.float64)
print(ad.shape, list(ad.obs.columns))
assert "batch" in ad.obs and ad.obs["batch"].nunique() >= 1
run_info = {"smoke_test": SMOKE_TEST, "n_cells": int(n), "n_genes": int(ad.n_vars), "obs_columns": list(ad.obs.columns),
            "cellcharter": "not run", "skipped": {}, "notes": []}
if not any(c in ad.obs.columns for c in ["fov", "FOV", "sample"]):
    run_info["notes"].append("no FOV/sample column used; batch is the only grouping")

(18895, 960) ['cell_type', 'niche', 'cx', 'cy', 'cx_g', 'cy_g', 'merge_cell_type', 'celltype_refined', 'batch', 'region', 'Region']


In [6]:
def run_cellcharter(ad):
    import scvi, squidpy as sq, cellcharter as cc
    scvi.settings.seed = SEED
    a = sc.AnnData(ad.layers["counts"].astype(np.float32).copy(), obs=ad.obs[["batch"]].copy())
    a.obs["batch"] = a.obs["batch"].astype("category")
    a.layers["counts"] = a.X.copy()
    a.obsm["spatial_global"] = np.asarray(ad.obsm["spatial_global"])
    scvi.model.SCVI.setup_anndata(a, layer="counts", batch_key="batch")
    model = scvi.model.SCVI(a, n_latent=CC_N_LATENT)
    model.train(max_epochs=SCVI_MAX_EPOCHS, early_stopping=True)
    a.obsm["X_scVI"] = model.get_latent_representation()
    sq.gr.spatial_neighbors(a, spatial_key="spatial_global", library_key="batch", coord_type="generic", n_neighs=CC_KNN)
    cc.gr.aggregate_neighbors(a, n_layers=CC_N_LAYERS, use_rep="X_scVI", out_key="X_cellcharter")
    autok = cc.tl.ClusterAutoK(n_clusters=CC_K_RANGE, max_runs=CC_MAX_RUNS, random_state=SEED)
    autok.fit(a, use_rep="X_cellcharter")
    lab = np.asarray(autok.predict(a, use_rep="X_cellcharter")).astype(str)
    return pd.Series(lab, index=a.obs_names), {"best_k": int(autok.best_k), "n_latent": CC_N_LATENT, "n_layers": CC_N_LAYERS}

def load_m1(ad):
    m1 = pd.read_csv(M1_PATH, index_col=0)["niche"].astype(str)
    out = m1.reindex(ad.obs_names)
    assert out.notna().all(), "M1 labels do not cover all cells"
    return out

runs = [("banksy", f"banksy_lam{l}_seed{s}", l, s) for l in BANKSY_LAMBDAS for s in BANKSY_SEEDS]
if SMOKE_TEST: runs = [runs[0], runs[-1]]
if USE_CELLCHARTER: runs.append(("cellcharter", "cellcharter_autok", None, None))
if USE_M1_PARTITION and os.path.exists(M1_PATH): runs.append(("m1", "m1_composition", None, None))
if SMOKE_TEST: runs = runs[:3]

bp_path = f"{out_dir}/base_partitions.csv"
if os.path.exists(bp_path):
    bp = pd.read_csv(bp_path, index_col=0, dtype=str)
    assert bp.index.equals(ad.obs_names), "saved base_partitions.csv does not match current cells; use a new OUT_DIR"
else:
    bp = pd.DataFrame(index=ad.obs_names)

n_b = sum(r[0] == "banksy" for r in runs)
est = max(1, (n_b * 4 + (12 if USE_CELLCHARTER else 0) + 15) * n / 18895)
print(f"Estimated runtime: ~{est:.0f} min ({n_b} BANKSY runs, CellCharter={USE_CELLCHARTER}, consensus included)")

for kind, name, lam, s in runs:
    if name in bp.columns:
        print("skip (saved):", name); continue
    t = time.time()
    try:
        if kind == "banksy":
            lab, _ = run_banksy(ad, K_GEOM, lam, BANKSY_RES, s)
        elif kind == "cellcharter":
            lab, info = run_cellcharter(ad); run_info["cellcharter"] = info
        else:
            lab = load_m1(ad)
    except Exception as e:
        if kind == "banksy": raise
        run_info["skipped"][name] = repr(e); run_info["cellcharter" if kind == "cellcharter" else name] = f"skipped: {e!r}"
        print("SKIPPED", name, repr(e)); continue
    bp[name] = lab.astype(str).values
    bp.to_csv(bp_path)
    print(f"done {name}: {bp[name].nunique()} clusters, {time.time() - t:.0f}s")
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

Estimated runtime: ~63 min (9 BANKSY runs, CellCharter=True, consensus included)

Median distance to closest cell = 44.01136216933077

---- Ran median_dist_to_nearest_neighbour in 0.04 s ----

---- Ran generate_spatial_distance_graph in 0.13 s ----

---- Ran row_normalize in 0.06 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 0.85 s ----

---- Ran generate_spatial_distance_graph in 0.24 s ----

---- Ran theta_from_spatial_graph in 0.21 s ----

---- Ran row_normalize in 0.07 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 1.21 s ----

Runtime Nov-25-2026-13-07

960 genes to be analysed:
Gene List:
Index(['AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1', 'ACKR3', 'ACKR4',
       'ACTA2', 'ACTG2',
       ...
       'WNT5B', 'WNT7A', 'WNT7B', 'WNT9A', 'XBP1', 'XCL1', 'XCL2', 'YBX3',
       'YES1', 'ZFP36'],
      dtype='str', length=960)

Check if X contains only finite (non-NAN) values
Decay Type: scaled_gaussian
Weights Object: {'weights': {0: <Compressed Sparse Row sparse

,decay,lambda_param,num_pcs,resolution,num_labels,labels,adata
scaled_gaussian_pc50_nc0.20_r0.30,scaled_gaussian,0.2,50,0.3,6,"Label object:\nNumber of labels: 6, number of ...",AnnData object with n_obs × n_vars = 18895 × 2...


done banksy_lam0.2_seed0: 6 clusters, 77s

Median distance to closest cell = 44.01136216933077

---- Ran median_dist_to_nearest_neighbour in 0.11 s ----

---- Ran generate_spatial_distance_graph in 0.33 s ----

---- Ran row_normalize in 0.13 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 1.75 s ----

---- Ran generate_spatial_distance_graph in 0.31 s ----

---- Ran theta_from_spatial_graph in 0.50 s ----

---- Ran row_normalize in 0.19 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 2.36 s ----

Runtime Nov-25-2026-13-09

960 genes to be analysed:
Gene List:
Index(['AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1', 'ACKR3', 'ACKR4',
       'ACTA2', 'ACTG2',
       ...
       'WNT5B', 'WNT7A', 'WNT7B', 'WNT9A', 'XBP1', 'XCL1', 'XCL2', 'YBX3',
       'YES1', 'ZFP36'],
      dtype='str', length=960)

Check if X contains only finite (non-NAN) values
Decay Type: scaled_gaussian
Weights Object: {'weights': {0: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 566850

,decay,lambda_param,num_pcs,resolution,num_labels,labels,adata
scaled_gaussian_pc50_nc0.20_r0.30,scaled_gaussian,0.2,50,0.3,5,"Label object:\nNumber of labels: 5, number of ...",AnnData object with n_obs × n_vars = 18895 × 2...


done banksy_lam0.2_seed1: 5 clusters, 76s

Median distance to closest cell = 44.01136216933077

---- Ran median_dist_to_nearest_neighbour in 0.04 s ----

---- Ran generate_spatial_distance_graph in 0.12 s ----

---- Ran row_normalize in 0.07 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 0.78 s ----

---- Ran generate_spatial_distance_graph in 0.22 s ----

---- Ran theta_from_spatial_graph in 0.22 s ----

---- Ran row_normalize in 0.06 s ----

---- Ran generate_spatial_weights_fixed_nbrs in 1.19 s ----

Runtime Nov-25-2026-13-10

960 genes to be analysed:
Gene List:
Index(['AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1', 'ACKR3', 'ACKR4',
       'ACTA2', 'ACTG2',
       ...
       'WNT5B', 'WNT7A', 'WNT7B', 'WNT9A', 'XBP1', 'XCL1', 'XCL2', 'YBX3',
       'YES1', 'ZFP36'],
      dtype='str', length=960)

Check if X contains only finite (non-NAN) values
Decay Type: scaled_gaussian
Weights Object: {'weights': {0: <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 566850

,decay,lambda_param,num_pcs,resolution,num_labels,labels,adata
scaled_gaussian_pc50_nc0.20_r0.30,scaled_gaussian,0.2,50,0.3,6,"Label object:\nNumber of labels: 6, number of ...",AnnData object with n_obs × n_vars = 18895 × 2...


... (15 outputs left out)

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=200` reached.


INFO     Creating graph using `None` transform and `4` libraries.                                                  


  0%|          | 0/4 [00:00<?, ?it/s]

SKIPPED cellcharter_autok TypeError("ClusterAutoK.__init__() got an unexpected keyword argument 'random_state'")


In [7]:
assert bp.shape[1] >= 1
base_stats = {}
for c in bp.columns:
    m, _, _, _ = evaluate(bp[c].values, ad.obs["Region"].values)
    base_stats[c] = {"n_clusters": int(bp[c].nunique()), **m}
pd.DataFrame(base_stats).T

,n_clusters,ARI,NMI
banksy_lam0.2_seed0,6.0,0.136737,0.230380
banksy_lam0.2_seed1,5.0,0.142232,0.232726
banksy_lam0.2_seed42,6.0,0.137197,0.233252
banksy_lam0.5_seed0,4.0,0.224881,0.309512
banksy_lam0.5_seed1,5.0,0.175313,0.273266
banksy_lam0.5_seed42,6.0,0.245011,0.325888
banksy_lam0.8_seed0,7.0,0.295289,0.360134
banksy_lam0.8_seed1,7.0,0.164412,0.307401
banksy_lam0.8_seed42,6.0,0.198779,0.326191


In [8]:
P = np.column_stack([pd.factorize(bp[c])[0] for c in bp.columns])
R = P.shape[1]
batch = ad.obs["batch"].values
knn = spatial_knn(coords, batch, CONSENSUS_KNN)
sym = sp.triu(knn.maximum(knn.T), k=1).tocoo()
er, ec = sym.row, sym.col
ew = ((P[er] == P[ec]).sum(1) / R).astype(np.float32)  # co-association per edge
g = ig.Graph(n=n, edges=list(zip(er.tolist(), ec.tolist())), directed=False)
g.es["weight"] = ew.tolist()
print(f"graph: {n} nodes, {len(ew)} edges, {R} base partitions, mean weight {ew.mean():.3f}")

def leiden_run(graph, res, seed):
    p = la.find_partition(graph, la.RBConfigurationVertexPartition, weights="weight", resolution_parameter=res, n_iterations=LEIDEN_ITERS, seed=seed)
    return np.array(p.membership)

# Label-free rule: among resolutions giving K_MIN-K_MAX niches (>= min size), pick the highest mean pairwise bootstrap ARI; never uses Region
min_n = int(np.ceil(MIN_NICHE_FRAC * n))
rng = np.random.RandomState(SEED)
full, boots, rows_curve = {}, {}, []
for res in RES_CANDIDATES:
    full[res] = leiden_run(g, res, SEED)
    bl = []
    for b in range(N_BOOT):
        idx = np.sort(rng.choice(n, int(BOOT_FRAC * n), replace=False))
        bl.append((idx, leiden_run(g.induced_subgraph(idx.tolist()), res, SEED + b + 1)))
    boots[res] = bl
    aris = []
    for (i1, l1), (i2, l2) in itertools.combinations(bl, 2):
        _, a1, a2 = np.intersect1d(i1, i2, return_indices=True)
        aris.append(ari_score(l1[a1], l2[a2]))
    cnt = np.bincount(full[res])
    rows_curve.append({"resolution": res, "n_clusters_raw": int(len(cnt)), "n_niches": int((cnt >= min_n).sum()), "mean_boot_ari": float(np.mean(aris))})
    print(rows_curve[-1])

curve = pd.DataFrame(rows_curve)
curve["eligible"] = curve["n_niches"].between(K_MIN, K_MAX)
if curve["eligible"].any():
    sel = float(curve[curve["eligible"]].sort_values("mean_boot_ari", ascending=False)["resolution"].iloc[0]); fallback = False
else:
    dist = np.where(curve["n_niches"] < K_MIN, K_MIN - curve["n_niches"], curve["n_niches"] - K_MAX)
    sel = float(curve.assign(d=dist).sort_values(["d", "mean_boot_ari"], ascending=[True, False])["resolution"].iloc[0]); fallback = True
    run_info["notes"].append("no resolution gave 4-12 niches; chose the one closest to the range")
curve["selected"] = curve["resolution"] == sel
curve.to_csv(f"{out_dir}/resolution_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve["resolution"], curve["mean_boot_ari"], "o-", label="mean bootstrap ARI"); ax.set_xlabel("resolution"); ax.set_ylabel("mean bootstrap ARI")
ax2 = ax.twinx(); ax2.plot(curve["resolution"], curve["n_niches"], "s--", color="gray", label="n niches"); ax2.axhspan(K_MIN, K_MAX, color="gray", alpha=0.1); ax2.set_ylabel("n niches")
ax.axvline(sel, color="red", ls=":"); ax.set_title(f"Resolution selection (chosen {sel})"); fig.tight_layout(); fig.savefig(f"{out_dir}/resolution_curve.png", dpi=150); plt.close(fig)
print("selected resolution:", sel)

graph: 18895 nodes, 509884 edges, 9 base partitions, mean weight 0.648
{'resolution': 0.1, 'n_clusters_raw': 8, 'n_niches': 8, 'mean_boot_ari': 0.8017673025572798}
{'resolution': 0.2, 'n_clusters_raw': 11, 'n_niches': 11, 'mean_boot_ari': 0.7524393551154047}
{'resolution': 0.3, 'n_clusters_raw': 16, 'n_niches': 16, 'mean_boot_ari': 0.7129986971237877}
{'resolution': 0.4, 'n_clusters_raw': 18, 'n_niches': 18, 'mean_boot_ari': 0.7083731178152611}
{'resolution': 0.5, 'n_clusters_raw': 22, 'n_niches': 22, 'mean_boot_ari': 0.6844312908949864}
{'resolution': 0.6, 'n_clusters_raw': 22, 'n_niches': 22, 'mean_boot_ari': 0.7012943780764519}
{'resolution': 0.7, 'n_clusters_raw': 28, 'n_niches': 28, 'mean_boot_ari': 0.6920613999861626}
{'resolution': 0.8, 'n_clusters_raw': 28, 'n_niches': 28, 'mean_boot_ari': 0.6982341852626527}
{'resolution': 0.9, 'n_clusters_raw': 31, 'n_niches': 31, 'mean_boot_ari': 0.6966042385865294}
{'resolution': 1.0, 'n_clusters_raw': 32, 'n_niches': 32, 'mean_boot_ari': 0

In [9]:
def merge_small(lab, er, ec, ew, n, min_n):
    W = sp.coo_matrix((np.r_[ew, ew], (np.r_[er, ec], np.r_[ec, er])), shape=(n, n)).tocsr()
    lab, log = lab.copy(), []
    while True:
        cnt = np.bincount(lab)
        small = [c for c in np.nonzero(cnt)[0] if cnt[c] < min_n]
        if not small or (cnt > 0).sum() == 1: break
        c = min(small, key=lambda x: cnt[x])
        Y = sp.csr_matrix((np.ones(n), (np.arange(n), lab)), shape=(n, len(cnt)))
        row = (Y.T @ W @ Y).toarray()[c].copy()
        row[c] = -1; row[cnt == 0] = -1
        tgt = int(np.argmax(row)) if row.max() > 0 else int(np.argmax(np.where(np.arange(len(cnt)) == c, -1, cnt)))
        log.append({"merged": int(c), "into": tgt, "n_cells": int(cnt[c])}); lab[lab == c] = tgt
    return lab, log

lab_raw = full[sel].copy()
lab_m, merge_log = merge_small(lab_raw, er, ec, ew, n, min_n)
order = pd.Series(lab_m).value_counts().index
remap = {old: str(i) for i, old in enumerate(order)}
niche = np.array([remap[x] for x in lab_m])
n_niches = len(order)
print("niches:", n_niches, "merges:", merge_log)

r2, c2, w2 = np.r_[er, ec], np.r_[ec, er], np.r_[ew, ew]
same = niche[r2] == niche[c2]
num = np.bincount(r2[same], weights=w2[same], minlength=n); den = np.bincount(r2[same], minlength=n)
cell_stab = np.where(den > 0, num / np.maximum(den, 1), 0.0)  # cells with no same-niche neighbour get 0

jac = {k: [] for k in np.unique(niche)}
for idx, bl in boots[sel]:
    ct_b = pd.crosstab(niche[idx], bl).values.astype(float)
    rs, cs = ct_b.sum(1), ct_b.sum(0)
    best = (ct_b / (rs[:, None] + cs[None, :] - ct_b)).max(1)
    for k, v in zip(sorted(np.unique(niche[idx])), best): jac[k].append(v)

stab = pd.DataFrame({"niche": sorted(np.unique(niche), key=int)})
stab["n_cells"] = [int((niche == k).sum()) for k in stab["niche"]]
stab["mean_stability"] = [float(cell_stab[niche == k].mean()) for k in stab["niche"]]
stab["min_stability"] = [float(cell_stab[niche == k].min()) for k in stab["niche"]]
stab["bootstrap_jaccard"] = [float(np.mean(jac[k])) if jac[k] else np.nan for k in stab["niche"]]
stab.to_csv(f"{out_dir}/niche_stability.csv", index=False)
stab

niches: 8 merges: []


,niche,n_cells,mean_stability,min_stability,bootstrap_jaccard
0,0,4497,0.581805,0.037665,0.836238
1,1,3276,0.598677,0.139434,0.851864
2,2,2902,0.631173,0.177778,0.959145
3,3,2726,0.690698,0.177778,0.833970
4,4,2240,0.704158,0.240079,0.722107
5,5,1617,0.873520,0.192872,0.980048
6,6,1114,0.670692,0.234801,0.610501
7,7,523,0.884573,0.301994,0.610801


In [10]:
mk_long = top10_markers(ad, niche, TOP_N_MARKERS)
m_cons, ct, maj, pur = evaluate(niche, ad.obs["Region"].values)
summary = pd.DataFrame({"niche": ct.index, "n_cells": ct.sum(1).values})
summary["frac_cells"] = summary["n_cells"] / n
summary["majority_region"] = maj.values; summary["purity"] = pur.values
summary = summary.merge(stab[["niche", "mean_stability"]], on="niche")

sc.pp.pca(ad, n_comps=50, random_state=SEED)
sil = float(silhouette_score(ad.obsm["X_pca"], niche, sample_size=min(5000, n), random_state=SEED)) if n_niches > 1 else None
sd = [ari_score(lab_raw, leiden_run(g, sel, SEED + 100 + i)) for i in range(2)]  # Leiden seed robustness at final resolution

import importlib.metadata as md
def _v(p):
    try: return md.version(p)
    except Exception: return None
libs = {p: _v(p) for p in ["scanpy", "anndata", "numpy", "pandas", "scipy", "scikit-learn", "torch", "igraph", "leidenalg", "harmony-pytorch", "scvi-tools", "cellcharter", "squidpy"]}

metrics = {
    "n_niches": int(n_niches), **m_cons, "silhouette_expression_pca": sil, "baseline_reference": BASELINE,
    "per_niche": {r["niche"]: {"n_cells": int(r["n_cells"]), "majority_region": r["majority_region"], "purity": float(r["purity"])} for _, r in summary.iterrows()},
    "stability": {"mean_cell_stability": float(cell_stab.mean()), "mean_niche_jaccard": float(stab["bootstrap_jaccard"].mean()),
                  "bootstrap_ari_at_selected": float(curve.loc[curve["selected"], "mean_boot_ari"].iloc[0]), "leiden_seed_ari": sd},
    "selection_record": {"rule": "highest mean pairwise bootstrap ARI among resolutions with 4-12 niches; label-free", "selected_resolution": sel,
                         "fallback_used": fallback, "candidates": curve.to_dict("records")},
    "base_partitions": base_stats, "n_base_partitions": int(R), "merge_log": merge_log,
}
config = {"CONFIG": CFG, "seed": SEED, "out_dir": out_dir, "library_versions": libs, "base_partitions": list(bp.columns),
          "min_niche_cells": min_n, "runtime_min": round((time.time() - t_start) / 60, 1), "gpu": torch.cuda.is_available(), **run_info}
save_contract(out_dir, ad, niche, {"cell_stability": cell_stab}, mk_long, summary, ct, metrics, config)
print(f"niches={n_niches} ARI={m_cons['ARI']:.3f} NMI={m_cons['NMI']:.3f} (baseline ARI {BASELINE['ARI']}, NMI {BASELINE['NMI']})")
summary

niches=8 ARI=0.137 NMI=0.280 (baseline ARI 0.248, NMI 0.323)


,niche,n_cells,frac_cells,majority_region,purity,mean_stability
0,0,4497,0.237999,myeloid-enriched stroma,0.585946,0.581805
1,1,3276,0.173379,stroma,0.643468,0.598677
2,2,2902,0.153586,myeloid-enriched stroma,0.674018,0.631173
3,3,2726,0.144271,myeloid-enriched stroma,0.338958,0.690698
4,4,2240,0.118550,myeloid-enriched stroma,0.316071,0.704158
5,5,1617,0.085578,tumor interior,0.677180,0.873520
6,6,1114,0.058957,stroma,0.681329,0.670692
7,7,523,0.027679,tumor interior,0.537285,0.884573


In [11]:
import shutil
zp = shutil.make_archive(out_dir, "zip", out_dir)
from google.colab import files
files.download(zp)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>